# 01 · Ground truth: COGERH lab results → `samples.csv`

COGERH publishes one trophic-state (IET) report per quarterly campaign. Each report has:
- **Table 3**: the lab results per reservoir (sample date, total N, total P, chlorophyll-a, cyanobacteria cells/mL, transparency);
- **Table 2**: stored volume (%) and trophic class.

Source: the official list at https://www.hidro.ce.gov.br/documentos/152 (28 reports, Feb/2016 to Feb/2025; see `data/raw/cogerh/SOURCES.md`). The reports changed layout over the years (2016-2021 vs 2022+), and the parser handles both.

This notebook reads every PDF in `data/raw/cogerh/`, checks the parser against four values copied from the Nov/2020 report, logs every row that could not be read, and writes `data/processed/samples.csv`.

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from common import RAW, PROCESSED, CYANO_THRESHOLD
from parse_cogerh import ParseLog, parse_report

pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_colwidth", 60)
pdfs = sorted((RAW / "cogerh").glob("*.pdf"))
[p.name for p in pdfs]

['iet_2016_02.pdf',
 'iet_2016_05.pdf',
 'iet_2016_08.pdf',
 'iet_2016_11.pdf',
 'iet_2017_02.pdf',
 'iet_2017_05.pdf',
 'iet_2017_08.pdf',
 'iet_2017_11.pdf',
 'iet_2018_02.pdf',
 'iet_2018_05.pdf',
 'iet_2018_08.pdf',
 'iet_2018_11.pdf',
 'iet_2019_02.pdf',
 'iet_2019_05.pdf',
 'iet_2019_08.pdf',
 'iet_2019_11.pdf',
 'iet_2020_02.pdf',
 'iet_2020_08.pdf',
 'iet_2020_11.pdf',
 'iet_2021_02.pdf',
 'iet_2021_05.pdf',
 'iet_2021_08.pdf',
 'iet_2021_11.pdf',
 'iet_2022_02.pdf',
 'iet_2022_11.pdf',
 'iet_2024_02.pdf',
 'iet_2024_08.pdf',
 'iet_2025_02.pdf']

## Parse every report

The PDFs have no ruled tables, so we read the text layer with `pdfplumber`. Every number is assigned to a column by its **horizontal position** under the column header. This matters because some cells are empty: reading the row left to right would shift the remaining values into the wrong columns. Rows we cannot read unambiguously are logged, never guessed. Values printed as `< x` (below the lab's detection limit) stay empty and keep the bound in the `censored` column.

In [2]:
log = ParseLog()
samples = pd.concat([parse_report(p, log) for p in pdfs], ignore_index=True)
print(samples.shape)
samples.head()

(3679, 15)


,reservoir_raw,date,page,censored,n_total,p_total,chla,cyano,transparency,campaign,file,reservoir,volume_pct,trophic_class,qc_flag
0,Acarape do Meio,2016-02-16,11,,1.473,0.056,35.33,208133.0,0.7,2016-02,iet_2016_02.pdf,acarape do meio,26.3,Eutrófica,
1,Amanary,2016-02-23,11,,3.400,0.471,48.06,108860.0,0.4,2016-02,iet_2016_02.pdf,amanary,4.7,Hipereutrófica,
2,Angicos,2016-02-25,11,,2.212,0.092,133.02,345220.0,0.4,2016-02,iet_2016_02.pdf,angicos,11.5,Eutrófica,
3,Aracoiaba,2016-02-16,11,,1.343,0.045,36.31,319067.0,0.9,2016-02,iet_2016_02.pdf,aracoiaba,20.7,Eutrófica,
4,Araras,2016-03-08,11,,4.162,0.198,199.42,359091.0,0.2,2016-02,iet_2016_02.pdf,araras,3.9,Hipereutrófica,


## Parser test

These four rows were copied by hand from Table 3 of the Nov/2020 report. The parser must reproduce them exactly.

In [3]:
expected = pd.DataFrame([
    ("castanhao",               "2020-11-19",  9.64,  32_209, 2.40),
    ("oros",                    "2020-11-18", 43.41, 313_603, 0.80),
    ("banabuiu",                "2020-11-04", 15.40,  60_880, 1.50),
    ("santo antonio de russas", "2020-11-24", 30.13, 138_927, 0.50),
], columns=["reservoir", "date", "chla", "cyano", "transparency"])

nov = samples[samples["campaign"] == "2020-11"].set_index("reservoir")
for _, e in expected.iterrows():
    got = nov.loc[e["reservoir"]]
    assert got["date"] == pd.Timestamp(e["date"]), (e["reservoir"], got["date"])
    assert got["chla"] == e["chla"], (e["reservoir"], got["chla"])
    assert got["cyano"] == e["cyano"], (e["reservoir"], got["cyano"])
    assert got["transparency"] == e["transparency"], (e["reservoir"], got["transparency"])
    print("PASS", e["reservoir"])

PASS castanhao
PASS oros
PASS banabuiu
PASS santo antonio de russas


## What was skipped or flagged, and why

Every line the parser did not turn into a clean sample is in `parse_log.csv`. Expected entries: page notes ("Continua", "Fonte"), reservoirs that were dry, cancelled or covered by macrophytes, and empty cells. Anything else is a real problem to look at.

In [4]:
lg = log.frame()
lg["kind"] = lg["reason"].str.replace(r"(:.*| '.*)$", "", regex=True)
print(lg.groupby(["kind", "file"]).size().unstack(fill_value=0))
lg.to_csv(PROCESSED / "parse_log.csv", index=False)

file                                                          iet_2016_02.pdf  iet_2016_05.pdf  iet_2016_08.pdf  iet_2016_11.pdf  iet_2017_02.pdf  iet_2017_05.pdf  iet_2017_08.pdf  iet_2017_11.pdf  \
kind                                                                                                                                                                                                   
campaign header differs from first page (2019-05)                           0                0                0                0                0                0                0                0   
cyano value repeated in another reservoir of the same cam...                2                2                4                4                2               12               16               12   
empty cells                                                                 4               34                7               15                0                0                6                5   


In [5]:
real = lg["kind"].str.startswith("unreadable") | (lg["kind"] == "no valid date") & ~lg["text"].str.contains("Fone")
lg[real][["file", "page", "text", "reason"]]

,file,page,text,reason
4,iet_2016_02.pdf,11.0,"Capitão Mor 16/02/16 4,975 0,149 270,97 870.609 ---",unreadable transparency value '---' -> NaN
9,iet_2016_02.pdf,11.0,"Cedro 17/02/16 5,738 0,336 215,70 22.285 ---",unreadable transparency value '---' -> NaN
30,iet_2016_02.pdf,13.0,"Pesqueiro 28/01/16 --- 0,101 --- 1.762.512 0,30",unreadable n_total value '---' -> NaN
31,iet_2016_02.pdf,13.0,"Pesqueiro 28/01/16 --- 0,101 --- 1.762.512 0,30",unreadable chla value '---' -> NaN
49,iet_2016_02.pdf,13.0,"Tigre 13/01/16 1,150 0,046 2,76 157.329 ---",unreadable transparency value '---' -> NaN
641,iet_2017_08.pdf,11.0,"Do Coronel 22/08/17 --- --- --- 10.368 0,70",unreadable n_total value '---' -> NaN
642,iet_2017_08.pdf,11.0,"Do Coronel 22/08/17 --- --- --- 10.368 0,70",unreadable p_total value '---' -> NaN
643,iet_2017_08.pdf,11.0,"Do Coronel 22/08/17 --- --- --- 10.368 0,70",unreadable chla value '---' -> NaN
652,iet_2017_08.pdf,12.0,"Jatobá II 17/08/17 --- 0,054 4,58 5.443 1,80",unreadable n_total value '---' -> NaN
657,iet_2017_08.pdf,12.0,"Muquém 22/08/17 --- --- --- 5.070 3,00",unreadable n_total value '---' -> NaN


In [6]:
lg[lg["kind"] == "status"]["reason"].str.replace("status: ", "").value_counts()

reason
Reservatório seco                                                                 1093
Seco                                                                                55
Macrófitas                                                                          40
Reservatório coberto por macrófitas                                                 27
Impossibilidade técnica                                                             20
Presença de macrófitas                                                              17
Cancelado                                                                           14
Coberto por macrófitas                                                              11
Recursos Hídricos (Cogerh)                                                           9
Parque Iracema  CEP: 60824-140                                                      9
Reservatório cancelado                                                               9
Presença de macrofitas              

## Samples per campaign

Non-null counts per column, and the range of sample dates. Lab visits inside one campaign are spread over several weeks, so the real sample date (not the campaign month) is what we will match to satellite images.

In [7]:
samples.groupby("campaign").agg(
    rows=("reservoir", "size"), cyano=("cyano", "count"), chla=("chla", "count"),
    n_total=("n_total", "count"), p_total=("p_total", "count"),
    transparency=("transparency", "count"), volume=("volume_pct", "count"),
    first_date=("date", "min"), last_date=("date", "max"))

,rows,cyano,chla,n_total,p_total,transparency,volume,first_date,last_date
campaign,,,,,,,,,
2016-02,111,111,110,110,111,108,111,2016-01-13,2016-04-06
2016-05,115,114,115,114,114,82,115,2016-02-04,2016-06-22
2016-08,114,114,108,114,113,114,114,2016-02-04,2016-09-28
2016-11,110,95,110,110,110,110,110,2016-02-04,2016-12-27
2017-02,99,99,99,99,99,99,99,2016-02-04,2017-03-28
2017-05,126,126,126,126,126,126,126,2016-02-04,2017-06-28
2017-08,118,118,114,114,114,118,118,2016-02-04,2017-11-09
2017-11,113,112,110,112,113,113,113,2016-02-04,2017-12-27
2018-02,110,110,105,109,109,110,110,2018-01-08,2018-03-27


## Class balance for the target

`y = 1` when cyanobacteria exceed **20,000 cells/mL** (Portaria GM/MS 888/2021, Annex XX, art. 43: above this, weekly cyanotoxin testing is required).

In [8]:
lab = samples.dropna(subset=["cyano"]).copy()
lab["y"] = (lab["cyano"] > CYANO_THRESHOLD).astype(int)
bal = lab.groupby("campaign")["y"].agg(n="size", positives="sum", share="mean").round(3)
print("all campaigns:", len(lab), "samples,", int(lab["y"].sum()), "positives, share", round(lab["y"].mean(), 3))
bal

all campaigns: 3652 samples, 2931 positives, share 0.803


,n,positives,share
campaign,,,
2016-02,111,108,0.973
2016-05,114,103,0.904
2016-08,114,91,0.798
2016-11,95,79,0.832
2017-02,99,74,0.747
2017-05,126,79,0.627
2017-08,118,61,0.517
2017-11,112,80,0.714
2018-02,110,76,0.691


## Quality flags

Two kinds of row are flagged in `qc_flag` (kept, not deleted):

- `cyano_repeated`: the same count appears in another reservoir of the same campaign. The database check below shows most of these are real values.
- `date_far_from_campaign`: the sample date is more than 75 days from the campaign month. Pedras Brancas keeps the date 04/02/2016 in seven campaigns while its values change every time, so the date was copied forward in the report. Satellite matching needs the true date, so these rows will not be matched to images.

In [9]:
print(samples["qc_flag"].replace("", "none").value_counts())
samples.loc[samples["qc_flag"].str.contains("date_far"), ["campaign", "reservoir_raw", "date", "cyano", "chla"]]

qc_flag
none                                     3427
cyano_repeated                            244
date_far_from_campaign                      7
cyano_repeated date_far_from_campaign       1
Name: count, dtype: int64


,campaign,reservoir_raw,date,cyano,chla
184,2016-05,Pedras Brancas,2016-02-04,24883.0,6.21
299,2016-08,Pedras Brancas,2016-02-04,124064.0,1.00
410,2016-11,Pedras Brancas,2016-02-04,103069.0,2.61
514,2017-02,Pedras Brancas,2016-02-04,45618.0,4.66
626,2017-05,Pedras Brancas,2016-02-04,8356.0,7.22
710,2017-08,Fogareiro,2017-11-09,15552.0,8.48
747,2017-08,Pedras Brancas,2016-02-04,283079.0,81.24
863,2017-11,Pedras Brancas,2016-02-04,4261.0,112.45


## Reservoirs across campaigns

How many reservoirs appear in every campaign. A name that appears only once may be a spelling difference that breaks the join.

In [10]:
per_res = samples.groupby("reservoir")["campaign"].nunique()
print(per_res.value_counts().sort_index().rename("campaigns -> reservoirs"))
print(sorted(per_res[per_res == 1].index))

campaign
4      2
5      1
6      1
7      1
9      2
10     2
11     4
13     2
14     2
15     6
16     2
17     3
18     2
19     3
20     2
21     5
22     5
23     8
24     5
25     5
26     8
27    21
28    64
Name: campaigns -> reservoirs, dtype: int64
[]


## Cross-check with COGERH's own database

The Portal Hidrológico do Ceará also serves the lab results as data (`src/fetch_hidro.py` saved them). It is a second official copy of the same measurements, so we compare it with what our parser read from the PDFs.

**One rule:** if the two sources disagree on a sample's date or cyanobacteria count, we do not use that sample. We do not try to decide which source is right.

**One exception**, found by hand: in Feb/2020 and Nov/2021 the database stores every count divided by 1,000 (46.132 instead of 46,132; counts above a million divided by 1,000,000), while dates and chlorophyll match exactly. That is the Brazilian thousands separator "." read as a decimal point. When the database value is the PDF value divided by exactly 1,000 or 1,000,000, we count the sources as agreeing.

The other big cause of disagreement is Aug/2019: the PDF prints Cauhipe's row twice and the rows after it carry their neighbour's values. Those samples are dropped by the rule.

In [11]:
from fetch_hidro import load_db_records
import numpy as np

db = load_db_records(PROCESSED.parent.parent)
m = samples.merge(db, on=["reservoir", "campaign"], how="left")
has = m["date_db"].notna()
date_differs = has & (m["date"] != m["date_db"])
both = has & m["cyano"].notna() & m["cyano_db"].notna()
same = np.isclose(m["cyano"].fillna(0), m["cyano_db"].fillna(0), rtol=1e-4, atol=0.5)
ratio = m["cyano"] / m["cyano_db"].replace(0, np.nan)
separator_slip = np.isclose(ratio, 1e3, rtol=1e-3) | np.isclose(ratio, 1e6, rtol=1e-3)
count_differs = both & ~same & ~separator_slip
samples["db_check"] = np.where(~has, "no_record", np.where(date_differs | count_differs, "disagree",
                               np.where(both & separator_slip, "agree_separator_slip", "agree")))
print(samples["db_check"].value_counts().to_string())
print(f"agreement where both have a value: count {1 - count_differs[both].mean():.3f}, date {1 - date_differs[has].mean():.3f}")
rep = samples["qc_flag"].str.contains("cyano_repeated")
print(f"repeated counts confirmed by the database: {(rep & (samples['db_check'] == 'agree')).sum()} of {rep.sum()}")
samples[samples["db_check"] == "disagree"].groupby("campaign").size().rename("disagreements").to_frame().T

db_check
agree                   2991
no_record                431
agree_separator_slip     234
disagree                  23
agreement where both have a value: count 0.996, date 0.993
repeated counts confirmed by the database: 201 of 245


campaign,2016-05,2016-08,2016-11,2017-02,2017-05,2017-08,2017-11,2018-02,2019-02,2019-08,2019-11,2021-05,2024-02
disagreements,1,1,1,1,1,1,2,1,1,10,1,1,1


## Which samples are usable

A sample is **usable** when it has a cyanobacteria count, its date is plausible for the campaign, and the database does not contradict it.

In [12]:
samples["usable"] = (samples["cyano"].notna()
                     & ~samples["qc_flag"].str.contains("date_far")
                     & (samples["db_check"] != "disagree"))
print(samples["usable"].sum(), "usable of", len(samples))
use = samples[samples["usable"]]
print("positive share among usable:", round((use["cyano"] > CYANO_THRESHOLD).mean(), 3))
use.groupby("campaign").size().rename("usable").to_frame().T

3628 usable of 3679
positive share among usable: 0.803


campaign,2016-02,2016-05,2016-08,2016-11,2017-02,2017-05,2017-08,2017-11,2018-02,2018-05,2018-08,2018-11,2019-02,2019-05,2019-08,2019-11,2020-02,2020-08,2020-11,2021-02,2021-05,2021-08,2021-11,2022-02,2022-11,2024-02,2024-08,2025-02
usable,111,113,113,94,98,125,116,110,109,139,140,136,131,139,125,128,125,141,142,144,142,141,142,134,141,149,151,149


## Save

In [13]:
cols = ["reservoir", "date", "chla", "cyano", "n_total", "p_total", "transparency", "volume_pct", "campaign",
        "usable", "db_check", "qc_flag", "reservoir_raw", "trophic_class", "censored", "file", "page"]
samples = samples.sort_values(["campaign", "reservoir"])[cols]
samples.to_csv(PROCESSED / "samples.csv", index=False, date_format="%Y-%m-%d")
print(samples.shape, "->", PROCESSED / "samples.csv")

(3679, 17) -> C:\Users\Filipe\alerta-acude\data\processed\samples.csv
